# Campagne CipherMark / WAF — Colab T4

Exécuter les cellules dans l'ordre. Les résultats sont sauvegardés dans Google Drive.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess

RACINE_DRIVE = "/content/drive/MyDrive/ciphermark"
DEPOT        = "/content/code-memoire"

# L'archive du Drive contient un dossier racine (code-memoire/) : la
# decompresser directement dans /content/code-memoire donnerait un niveau de
# trop, et "No module named distseal.utils". On extrait donc a cote, puis on
# localise le dossier qui contient reellement distseal/utils au lieu de le
# supposer -- c'est le bug qui faisait echouer ce carnet : la cellule
# precedente supposait /content/drive/MyDrive/code-memoire deja present, ce
# qui n'est jamais le cas.
if not os.path.isdir(os.path.join(DEPOT, "distseal", "utils")):
    arch = os.path.join(RACINE_DRIVE, "00-code-source-a-jour.tar.gz")
    if not os.path.exists(arch):
        raise FileNotFoundError(f"archive introuvable : {arch}")
    tmp = "/content/_extraction"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp, exist_ok=True)
    subprocess.run(["tar", "xzf", arch, "-C", tmp], check=True)
    racine = next((r for r, _, _ in os.walk(tmp)
                   if os.path.isdir(os.path.join(r, "distseal", "utils"))), None)
    if racine is None:
        raise SystemExit("l'archive ne contient pas distseal/utils")
    shutil.rmtree(DEPOT, ignore_errors=True)
    shutil.move(racine, DEPOT)

sys.path[:0] = [os.path.join(DEPOT, "deps"), DEPOT]
os.chdir(DEPOT)
print(f"code : {DEPOT}  (distseal/utils present)")

# --- les checkpoints, copies UNE FOIS depuis le Drive monte ------------------
# Le Drive est un systeme de fichiers monte : une simple copie suffit, pas
# besoin de rclone (reserve au pod RunPod, qui n'a pas ce montage).
os.makedirs("/content/ckpt", exist_ok=True)
CKPT      = "/content/ckpt/base_64bits.pth"
COND_DIFF = "/content/ckpt/phaseD512_degele.pt"
COND_AR   = "/content/ckpt/phaseF_maskgit.pt"

SOURCES = {
    CKPT:      f"{RACINE_DRIVE}/01-tatoueur-modeles-de-reference-64-et-128-bits/"
               f"phaseA2_64bits_stable_checkpoint.pth",
    COND_DIFF: f"{RACINE_DRIVE}/04-decodeur-generatif-conditionne-sur-omega-modele-retenu/"
               f"checkpoint.pt",
    COND_AR:   f"{RACINE_DRIVE}/06-entrainements-en-cours-non-termines/phaseF_maskgit/"
               f"checkpoint.pt",
}
for dst, src in SOURCES.items():
    if os.path.exists(dst):
        continue
    if not os.path.exists(src):
        raise FileNotFoundError(f"checkpoint absent du Drive : {src}")
    print(f"copie {os.path.getsize(src) / 2**20:.0f} Mo -> {dst}")
    shutil.copy(src, dst)
print("checkpoints prets :", CKPT, COND_DIFF, COND_AR, sep="\n  ")


In [ ]:
import importlib.util, subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "torchmetrics",
                "einops", "omegaconf", "safetensors"])

# Trois dependances ne se signalent PAS par un ModuleNotFoundError a
# l'import : elles sont rattrapees dans un try/except interne et ne
# provoquent une erreur qu'a l'USAGE, en pleine campagne.
#   - reedsolo         : phash.py la rattrape, echoue plus tard silencieusement.
#   - pycryptodome      : crypto.py bascule sinon sur un repli HMAC-CTR qui
#     produit un keystream DIFFERENT -- les marques deviennent inverifiables
#     ailleurs, SANS AUCUNE ERREUR. C'est le piege le plus grave des trois.
#   - huggingface_hub  : necessaire au chargement de DINOv2.
for mod, paquet in (("reedsolo", "reedsolo"), ("Crypto", "pycryptodome"),
                    ("huggingface_hub", "huggingface_hub")):
    if importlib.util.find_spec(mod) is None:
        print("installation de", paquet)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet])

# check=False + capture explicite : un CalledProcessError masque le message
# reel derriere sa trace Python, et le cwd peut avoir change entre deux
# cellules (redemarrage de session sans rejouer la cellule 2). On distingue
# donc "fichier introuvable" (rejouer la cellule 2) d'une vraie erreur de
# syntaxe (a signaler).
def verifie(chemin):
    if not os.path.exists(chemin):
        raise SystemExit(
            f"\n  {chemin} introuvable depuis {os.getcwd()}.\n"
            f"  La session a probablement redemarre : relancez la cellule 2\n"
            f"  (montage du Drive + recuperation du code) avant de continuer.\n")
    r = subprocess.run([sys.executable, "-m", "py_compile", chemin],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"\n  erreur de syntaxe dans {chemin} :\n{r.stderr}")
    print(f"  ok : {chemin}")

verifie("scripts/ciphermark/eval_grande_echelle_decodeur.py")
verifie("scripts/ciphermark/build_corpus.py")
print("Dependances et syntaxe verifiees.")


## Smoke test (à exécuter en premier)

In [ ]:
import os, subprocess

# Corpus d'EVALUATION, disjoint du corpus d'entrainement.
#
# ATTENTION : ne pas reutiliser corpus-colab.tar.gz du Drive -- c'est le
# corpus d'ENTRAINEMENT (5398 train / 599 val). L'utiliser ici pour
# l'evaluation reproduirait exactement la contamination corpus
# d'apprentissage / corpus d'evaluation identifiee et corrigee au chapitre 3
# ("Une contamination du corpus"). Le corpus d'evaluation se reconstruit donc
# depuis COCO test2017 -- le meme split que celui de la campagne du memoire,
# jamais vu par aucun modele entraine.
CORPUS_SMOKE = "/content/corpus-smoke"
if not os.path.isdir(CORPUS_SMOKE) or len(os.listdir(CORPUS_SMOKE)) < 120:
    subprocess.run(["python3", "scripts/ciphermark/build_corpus.py",
                    "--out", CORPUS_SMOKE, "--coco", "test2017",
                    "--coco-max", "120"], check=True)

os.environ["RACINE"]    = "/content/code-memoire"
os.environ["CKPT"]      = "/content/ckpt/base_64bits.pth"
os.environ["COND_DIFF"] = "/content/ckpt/phaseD512_degele.pt"
os.environ["COND_AR"]   = "/content/ckpt/phaseF_maskgit.pt"
os.environ["CORPUS"]    = CORPUS_SMOKE
os.environ["N"] = "100"; os.environ["N_GEN"] = "20"; os.environ["WITH_LPIPS"] = "0"
os.environ["SORTIE"] = "/content/drive/MyDrive/ciphermark_runs/smoke"
get_ipython().system("bash runpod/campagne_metriques_unifiees.sh")


## Campagne complète

Lancer seulement après un smoke test réussi. Les fichiers existants sont ignorés pour permettre la reprise.

In [ ]:
import os, subprocess

# Meme corpus disjoint que le smoke test, elargi a 12 000 scenes COCO
# test2017 -- la taille du corpus d'evaluation du memoire (12 025 images avec
# les 25 de skimage). Le telechargement de test2017.zip (~1 Go) prend
# quelques minutes ; il ne se refait pas si le dossier est deja complet.
CORPUS_FULL = "/content/corpus-eval-12k"
if not os.path.isdir(CORPUS_FULL) or len(os.listdir(CORPUS_FULL)) < 12000:
    subprocess.run(["python3", "scripts/ciphermark/build_corpus.py",
                    "--out", CORPUS_FULL, "--coco", "test2017",
                    "--coco-max", "12000"], check=True)

os.environ["RACINE"]    = "/content/code-memoire"
os.environ["CKPT"]      = "/content/ckpt/base_64bits.pth"
os.environ["COND_DIFF"] = "/content/ckpt/phaseD512_degele.pt"
os.environ["COND_AR"]   = "/content/ckpt/phaseF_maskgit.pt"
os.environ["CORPUS"]    = CORPUS_FULL
os.environ["N"] = "5000"; os.environ["N_GEN"] = "1000"; os.environ["WITH_LPIPS"] = "1"
os.environ["SORTIE"] = "/content/drive/MyDrive/ciphermark_runs/final"
get_ipython().system("bash runpod/campagne_metriques_unifiees.sh")


In [ ]:
from pathlib import Path
root=Path('/content/drive/MyDrive/ciphermark_runs')
for p in sorted(root.rglob('*.json')): print(p, p.stat().st_size//1024, 'Ko')